# Setup

### 01 Install required packages

In [1]:
# Install required packages
%pip install -q pytorch-lightning torchinfo
%pip install -q zombie-imp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 4.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 15.6 MB/s eta 0:00:00a 0:00:01


### 02 Clone the repository

In [2]:
# Clone the repository to Colab environment
!git clone https://github.com/ilyarudyak/DL_projects_2026.git

Cloning into 'DL_projects_2026'...
remote: Enumerating objects: 34, done.
remote: Counting objects: 100% (34/34), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 34 (delta 8), reused 34 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (34/34), 412.67 KiB | 2.55 MiB/s, done.
Resolving deltas: 100% (8/8), done.


### 03 Switch to the project directory

In [3]:
import os

# Move into your specific project folder on the remote machine
os.chdir("/content/DL_projects_2026/01-sentiment-analysis")

# Print the directory contents to verify your python modules (.py files) are there
print("Current Working Directory:", os.getcwd())
print("\n=== Available Project Files ===")
!ls -la

Current Working Directory: /content/DL_projects_2026/01-sentiment-analysis

=== Available Project Files ===
total 920
drwxr-xr-x 4 root root   4096 Jul 21 14:34 .
drwxr-xr-x 4 root root   4096 Jul 21 14:34 ..
-rw-r--r-- 1 root root 186973 Jul 21 14:34 14_nlp_with_rnns_and_attention.ipynb
-rw-r--r-- 1 root root 594808 Jul 21 14:34 14-sentiment_analysis.ipynb
-rw-r--r-- 1 root root  16944 Jul 21 14:34 bpe_tokenizer.py
-rw-r--r-- 1 root root   8828 Jul 21 14:34 byte_bpe_tokenizer.py
-rw-r--r-- 1 root root  10547 Jul 21 14:34 bytes.ipynb
-rw-r--r-- 1 root root   1812 Jul 21 14:34 changes_in_trainer.md
drwxr-xr-x 2 root root   4096 Jul 21 14:34 configs
-rw-r--r-- 1 root root  18264 Jul 21 14:34 dataset.py
-rw-r--r-- 1 root root   1705 Jul 21 14:34 logging.csv
-rw-r--r-- 1 root root  24608 Jul 21 14:34 model.py
drwxr-xr-x 2 root root   4096 Jul 21 14:34 printouts
-rw-r--r-- 1 root root  10568 Jul 21 14:34 sentiment_analysis_colab_v1.ipynb
-rw-r--r-- 1 root root   4987 Jul 21 14:34 sentiment_

### 04 Import libraries

In [4]:
%load_ext autoreload
%autoreload 2

from dataset import IMDBConfig, IMDBData
from model import IMDBModelLP, IMDBModelLPPackedSeq, IMDBModelLPV2
from train import TrainerHighLevel

# Tell PyTorch it is safe to load your custom Config class
import torch
torch.serialization.add_safe_globals([IMDBConfig, IMDBData, IMDBModelLP])

# Set up logging format and level
import logging
# logging.basicConfig(format="%(asctime)s - %(name)s - %(levelname)s - %(message)s")
logging.basicConfig(format="%(levelname)s:%(name)s:  %(message)s")

### 05 Set logging levels [OPTIONAL]

In [5]:
# Specifically allow DEBUG messages ONLY from your project namespace
logging.getLogger("imdb").setLevel(logging.DEBUG)

In [ ]:
# Specifically disallow DEBUG messages ONLY from your project namespace
logging.getLogger("imdb").setLevel(logging.INFO)

### 06 Check hardware specifications [OPTIONAL]

In [6]:
# Check VM OS, RAM, and available disk space
print("=== Operating System ===")
!lsb_release -a

print("\n=== CPU Specifications ===")
!lscpu | grep "Model name\|CPU(s):"

print("\n=== System RAM ===")
!free -h

print("\n=== Disk Space ===")
!df -h /

=== Operating System ===
No LSB modules are available.
Distributor ID:	Ubuntu
Description:	Ubuntu 22.04.5 LTS
Release:	22.04
Codename:	jammy

=== CPU Specifications ===
CPU(s):                                  2
Model name:                              Intel(R) Xeon(R) CPU @ 2.20GHz
NUMA node0 CPU(s):                       0,1

=== System RAM ===
               total        used        free      shared  buff/cache   available
Mem:            12Gi       1.0Gi       7.7Gi       2.0Mi       3.9Gi        11Gi
Swap:             0B          0B          0B

=== Disk Space ===
Filesystem      Size  Used Avail Use% Mounted on
overlay         108G   21G   88G  19% /


### 07 Verify GPU Availability [OPTIONAL]

In [7]:
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))
    print("CUDA Capability:", torch.cuda.get_device_capability(0))
else:
    print("Running on CPU.")

PyTorch Version: 2.11.0+cpu
CUDA Available: False
Running on CPU.


### 08 End the session [OPTIONAL]

In [ ]:
from google.colab import runtime
runtime.unassign()

### 09 Pull the latest changes from the repository [OPTIONAL]

In [ ]:
!git pull

# 01 IMDB Dataset

In [8]:
# Create config file for IMDB dataset
config_file = "configs/base_config.yaml"
config = IMDBConfig.from_yaml(config_file)

In [10]:
# Create an instance of the IMDBData class
data = IMDBData(config=config)

DEBUG:imdb.dataset:===DATASET LOADED===
DEBUG:imdb.dataset:Dataset splits: dict_keys(['train', 'test', 'unsupervised'])
DEBUG:imdb.dataset:Number of training samples (original dataset): 1000
DEBUG:imdb.dataset:Number of test samples (original dataset): 500
DEBUG:imdb.dataset:===DATASET SPLIT===
DEBUG:imdb.dataset:Train/val split ratios: 0.8/0.2
DEBUG:imdb.dataset:Number of training samples (after split): 800
DEBUG:imdb.dataset:Number of validation samples (after split): 200
DEBUG:imdb.dataset:===LOADING PRETRAINED BBPE TOKENIZER (gpt2)===
DEBUG:imdb.dataset:Using local tokenizer found at: datasets/tokenizers/gpt2/tokenizer.json


In [11]:
# Create loaders for the training and validation datasets
train_loader, val_loader = data.get_loaders()

DEBUG:imdb.dataset:===DATA LOADERS CREATED===
DEBUG:imdb.dataset:Train loader: 2 batches, 800 samples
DEBUG:imdb.dataset:Validation loader: 1 batches, 200 samples


# 02 Training the model

In [12]:
# Specify the config file
config_name = "base_config"

# Create an instance of a TrainerHighLevel with a toy dataset
trainer = TrainerHighLevel(config_name=config_name, 
                           data_limit=1000, # Use the toy dataset
                           model_class=IMDBModelLPPackedSeq,
                           device='auto'
                           )

# Fit the model using the trainer
trainer.fit()

# Plot the training and validation loss curves
trainer.plot_training_curves()

DEBUG:imdb.trainer:===CONFIG INITIALIZATION===
DEBUG:imdb.trainer:Config name: base_config
DEBUG:imdb.dataset:===DATASET LOADED===
DEBUG:imdb.dataset:Dataset splits: dict_keys(['train', 'test', 'unsupervised'])
DEBUG:imdb.dataset:Number of training samples (original dataset): 1000
DEBUG:imdb.dataset:Number of test samples (original dataset): 500
DEBUG:imdb.dataset:===DATASET SPLIT===
DEBUG:imdb.dataset:Train/val split ratios: 0.8/0.2
DEBUG:imdb.dataset:Number of training samples (after split): 800
DEBUG:imdb.dataset:Number of validation samples (after split): 200
DEBUG:imdb.dataset:===LOADING PRETRAINED BBPE TOKENIZER (gpt2)===
DEBUG:imdb.dataset:Using local tokenizer found at: datasets/tokenizers/gpt2/tokenizer.json
DEBUG:imdb.dataset:===DATA LOADERS CREATED===
DEBUG:imdb.dataset:Train loader: 2 batches, 800 samples
DEBUG:imdb.dataset:Validation loader: 1 batches, 200 samples
DEBUG:imdb.model:===MODEL CREATION===
DEBUG:imdb.model:Model initialized with vocab_size: 50259
DEBUG:imdb.mod

🚀 Using hardware accelerator: cpu


INFO:pytorch_lightning.utilities.rank_zero:Loading `train_dataloader` to estimate number of stepping batches.
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/loops/fit_loop.py:321: The number of training batches (2) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.
DEBUG:imdb.dataset:===COLLATE_FN===
DEBUG:imdb.dataset:Batch type: <class 'list'>, Batch size: 200
DEBUG:imdb.dataset:First item type: <class 'dict'>, Keys: dict_keys(['text', 'label'])
DEBUG:imdb.dataset:First item text length: 748, First item label: 0
DEBUG:imdb.dataset:Encodings type: <class 'list'>, Number of encodings: 200
DEBUG:imdb.dataset:First encoding type: <class 'tokenizers.Encoding'>
DEBUG:imdb.dataset

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/call.py", line 49, in _call_and_handle_interrupt
    return trainer_fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/trainer.py", line 630, in _fit_impl
    self._run(model, ckpt_path=ckpt_path, weights_only=weights_only)
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/trainer.py", line 1079, in _run
    results = self._run_stage()
              ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/trainer.py", line 1123, in _run_stage
    self.fit_loop.run()
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/loops/fit_loop.py", line 217, in run
    self.advance()
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/loops/fit_loop.py", line 469, in advance
    self.epoch_loop.run(self._data_fetcher)
  File

TypeError: object of type 'NoneType' has no len()